In [9]:
%pip install numpy pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../house-prices-advanced-regression-techniques/train.csv")

selected_features = ["GrLivArea", "OverallQual"]

X = df[selected_features].values
y = df["SalePrice"].values

print(X.shape, y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train)
X_scaled_test = scaler.transform(X_test)

(1460, 2) (1460,)


# Scratch

In [12]:
from itertools import combinations_with_replacement
from sklearn.metrics import mean_squared_error, r2_score

class PolyRegressionScratch:
    def __init__(self, degree=2, learning_rate=0.01, n_iterations=1000):
        self.degree = degree
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.powers = None

    def _expand_features(self, X):
        n_samples, n_features = X.shape
        self.powers = []
        for deg in range(1, self.degree + 1):
            for comb in combinations_with_replacement(range(n_features), deg):
                count = [0] * n_features
                for idx in comb:
                    count[idx] += 1
                self.powers.append(count)

        X_poly = np.ones((n_samples, 1))
        for power in self.powers:
            term = np.prod(X**power, axis=1).reshape(-1, 1)
            X_poly = np.hstack([X_poly, term])
        return X_poly

    def predict(self, X):
        X_poly = self._expand_features(X)
        return X_poly @ self.weights

    def compute_cost(self, X, y):
        m = len(y)
        y_pred = self.predict(X)
        return (1 / m) * np.sum((y_pred - y) ** 2)

    def fit(self, X, y):
        X_poly = self._expand_features(X)
        n_samples, n_features_poly = X_poly.shape
        self.weights = np.ones(n_features_poly)

        for i in range(self.n_iterations):
            y_pred = X_poly @ self.weights
            gradient = (2 / n_samples) * X_poly.T @ (y_pred - y)
            self.weights -= self.learning_rate * gradient

            if i % 100 == 0:
                y_pred = self.predict(X)
                mse = mean_squared_error(y, y_pred)
                r2 = r2_score(y, y_pred)
                print(f"Iteration {i}, MSE: {mse}, R^2: {r2}")

In [13]:
model = PolyRegressionScratch(degree=2, learning_rate=0.01, n_iterations=1001)
model.fit(X_scaled_train, y_train)

y_pred = model.predict(X_scaled_test)
r2_train = r2_score(y_train, model.predict(X_scaled_train))
r2_test = r2_score(y_test, y_pred)
print(f"R² train: {r2_train:.3f}")
print(f"R² test: {r2_test:.3f}")

mse_train = mean_squared_error(y_train, model.predict(X_scaled_train))
print(f"MSE train: {mse_train:.3f}")
mse_test = mean_squared_error(y_test, y_pred)
print(f"MSE test: {mse_test:.3f}")


Iteration 0, MSE: 32988669627.684074, R^2: -4.530788999547925
Iteration 100, MSE: 2973168154.821423, R^2: 0.5015268602801501
Iteration 200, MSE: 1713316027.9192016, R^2: 0.7127501791702319
Iteration 300, MSE: 1530157065.1419978, R^2: 0.743458103676731
Iteration 400, MSE: 1473524412.5521884, R^2: 0.7529529773862207
Iteration 500, MSE: 1449617145.511225, R^2: 0.75696120357575
Iteration 600, MSE: 1438816734.489719, R^2: 0.7587719705797705
Iteration 700, MSE: 1433867686.0681918, R^2: 0.7596017143335181
Iteration 800, MSE: 1431591180.8935676, R^2: 0.7599833868871351
Iteration 900, MSE: 1430542598.0070977, R^2: 0.7601591893902082
Iteration 1000, MSE: 1430059341.4165347, R^2: 0.7602402108519771
R² train: 0.760
R² test: 0.797
MSE train: 1430059341.417
MSE test: 1557747920.044


# Scikit-Learn

In [14]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("linreg", LinearRegression()),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.2f}")

R² Score: 0.7968
Mean Squared Error: 1558297034.29
